## Cell 1 — Imports & Logging

In [0]:
import re
import logging
import requests
import pandas as pd

from datetime import datetime, timezone
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, DoubleType, IntegerType

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s — %(message)s"
)
logger = logging.getLogger("stl_county_crime_ingest")

print("✓ Imports loaded")

## Cell 2 — Configuration

In [0]:
# Defining the output path
RAW_OUTPUT_PATH = "/Volumes/workspace/default/raw/crime/county"

print(f"✓ Output path: {RAW_OUTPUT_PATH}")

## Cell 3 — Dataset Registry

In [0]:
# ArcGIS Hub dataset IDs for STL County NIBRS crime data.
# Each year is a separate published dataset on data.stlouisco.com.
# Narrowed to 2024+ to match the city crime data date range.
# To add future years: find the dataset page on data.stlouisco.com,
# copy the ID from the URL, and add it here.
DATASET_REGISTRY: dict[int, str] = {
    2024: "9d588c75469741afbe22e58479a01fbf",
    2025: "26ebc7d48d5a42c7ad0b214f7f9352db",
}

# ArcGIS Hub GeoJSON endpoint — returns up to 1000 records per page.
# {dataset_id} is replaced at runtime. Layer index is always 0.
ARCGIS_URL_TEMPLATE = (
    "https://opendata.arcgis.com/datasets/{dataset_id}_0.geojson"
)

# ArcGIS Feature Service query endpoint — used for paginated full pulls.
# resultOffset and resultRecordCount control pagination.
ARCGIS_QUERY_TEMPLATE = (
    "https://services.arcgis.com/hifzdpUOCBiMtiCk/arcgis/rest/services/"
    "NIBRS_Crime_Data_{year}/FeatureServer/0/query"
    "?where=1%3D1&outFields=*&f=json"
    "&resultOffset={offset}&resultRecordCount={page_size}"
)

BROWSER_HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
}

print(f"✓ Dataset registry: {list(DATASET_REGISTRY.keys())}")

In [0]:
# From search results we can see STL County runs their own ArcGIS
# server at maps.stlouisco.com. Let's try querying the NIBRS
# service directly from there.

# Also try the standard ArcGIS Online hosted service pattern
urls_to_try = [
    # STL County's own ArcGIS server
    "https://maps.stlouisco.com/arcgis/rest/services/OpenData/NIBRS_Crime_Data_2024/FeatureServer/0/query?where=1=1&outFields=*&resultRecordCount=3&f=json",
    # ArcGIS Online hosted — common pattern for Hub datasets
    "https://services1.arcgis.com/hifzdpUOCBiMtiCk/arcgis/rest/services/NIBRS_Crime_Data_2024/FeatureServer/0/query?where=1=1&outFields=*&resultRecordCount=3&f=json",
    # Try the county's open data folder specifically
    "https://maps.stlouisco.com/arcgis/rest/services/OpenData/NIBRS2024/FeatureServer/0/query?where=1=1&outFields=*&resultRecordCount=3&f=json",
]

for url in urls_to_try:
    print(f"\nTrying: {url[:80]}...")
    try:
        resp = requests.get(url, headers=BROWSER_HEADERS, timeout=30)
        print(f"  Status: {resp.status_code}")
        if resp.status_code == 200:
            data = resp.json()
            if "error" in data:
                print(f"  ArcGIS error: {data['error'].get('message')}")
            else:
                features = data.get("features", [])
                print(f"  ✓ Features returned: {len(features)}")
                if features:
                    attrs = features[0].get("attributes", {})
                    print(f"  Fields: {list(attrs.keys())}")
                break
    except Exception as e:
        print(f"  Exception: {e}")